In [2]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Excel dosya yolları
YUKLER_DOSYASI = r"C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\SLF_YUK_DAGITIMI\hor_ver_yuk.xlsx"  # İlk ekran görüntüsündeki veriler
ABONELER_DOSYASI = r"C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\SLF_YUK_DAGITIMI\output_abone_dagilimi_kumulatif3.xlsx"  # İkinci ekran görüntüsündeki veriler
CIKTI_DOSYASI = r"C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\SLF_YUK_DAGITIMI\yukler_sonuc1.xlsx"  # Çıktı dosyası

# Abone tipleri
ABONE_TIPLERI = ["MESKEN", "SANAYI", "TICARETHANE", "TARIMSAL_SULAMA", "AYDINLATMA"]

# Yıllar
YILLAR = list(range(2024, 2036))  # 2024-2035 arası

def yuk_hesaplama():
    """Ana hesaplama fonksiyonu"""
    print("Yük hesaplama başlatılıyor...")
    
    # Yük verilerini oku
    yukler_df = pd.read_excel(YUKLER_DOSYASI)
    
    # Excel yazıcısı oluştur
    writer = pd.ExcelWriter(CIKTI_DOSYASI, engine='xlsxwriter')
    
    # Her yıl için hesaplama yap
    for yil in YILLAR:
        print(f"{yil} yılı için hesaplama yapılıyor...")
        
        # Önceki yıl - hesaplamalar için
        onceki_yil = yil - 1
        
        # Aboneleri oku
        try:
            # Mevcut yıl aboneleri
            current_year_sheet = str(yil)
            current_year_aboneler = pd.read_excel(ABONELER_DOSYASI, sheet_name=current_year_sheet)
            
            # Önceki yıl aboneleri
            previous_year_sheet = str(onceki_yil)
            if onceki_yil == 2023:
                previous_year_sheet = "As-is"  # 2023 değerleri "as-is" sheet'inde
            
            previous_year_aboneler = pd.read_excel(ABONELER_DOSYASI, sheet_name=previous_year_sheet)
            
            # Sonuç dataframe'ini oluştur - aynı format ile
            sonuc_df = current_year_aboneler.copy()
            
            # Abone sütunlarını sıfırla - sadece yapısını kullanacağız
            for abone_tipi in ABONE_TIPLERI:
                sonuc_df[abone_tipi] = 0.0
            
            # Her ilçe için hesaplama yap
            for ilce in sonuc_df['ilce'].unique():
                # Her abone tipi için hesaplama yap
                for abone_tipi in ABONE_TIPLERI:
                    # İlçeye göre filtreleme
                    current_year_ilce = current_year_aboneler[current_year_aboneler["ilce"] == ilce]
                    previous_year_ilce = previous_year_aboneler[previous_year_aboneler["ilce"] == ilce]
                    
                    # Yük verilerinden ilgili değerleri al
                    yuk_satiri = yukler_df[(yukler_df["YIL"] == yil) & (yukler_df["ILCE"] == ilce)]
                    onceki_yuk_satiri = yukler_df[(yukler_df["YIL"] == onceki_yil) & (yukler_df["ILCE"] == ilce)]
                    
                    if yuk_satiri.empty or onceki_yuk_satiri.empty:
                        print(f"UYARI: {yil} veya {onceki_yil} yılı için {ilce} ilçesinde yük verisi bulunamadı.")
                        continue
                    
                    # Toplam faturalanan yük ve vertical yük değerlerini al
                    faturalanan_yuk_kolonu = f"{abone_tipi}_FATURALANAN"
                    vertical_yuk_kolonu = f"{abone_tipi}_VERTICAL"
                    
                    toplam_faturalanan = yuk_satiri[faturalanan_yuk_kolonu].values[0]
                    toplam_vertical = yuk_satiri[vertical_yuk_kolonu].values[0]
                    
                    # Reel yükü hesapla (toplam - vertical)
                    reel_yuk = toplam_faturalanan - toplam_vertical
                    
                    # İlgili yıl için toplam abone sayısını hesapla
                    toplam_abone_sayisi = current_year_ilce[abone_tipi].sum()
                    onceki_yil_toplam_abone = previous_year_ilce[abone_tipi].sum()
                    
                    if toplam_abone_sayisi == 0:
                        print(f"UYARI: {yil} yılı için {ilce} ilçesinde {abone_tipi} abone sayısı 0, hesaplama yapılamadı.")
                        continue
                    
                    if onceki_yil_toplam_abone == 0:
                        print(f"UYARI: {onceki_yil} yılı için {ilce} ilçesinde {abone_tipi} abone sayısı 0, vertical yük hesaplanamadı.")
                        abone_basina_vertical_yuk = 0
                    else:
                        # Abone başına vertical yük
                        abone_basina_vertical_yuk = toplam_vertical / onceki_yil_toplam_abone
                    
                    # Abone başına reel yük
                    abone_basina_reel_yuk = reel_yuk / toplam_abone_sayisi
                    
                    # Her hücre için yük hesaplama
                    for hucre_idx, hucre in current_year_ilce.iterrows():
                        hucre_id = hucre["id"]
                        current_abone_sayisi = hucre[abone_tipi]
                        
                        # Önceki yıldaki aynı hücreyi bul
                        previous_hucre = previous_year_ilce[previous_year_ilce["id"] == hucre_id]
                        
                        if not previous_hucre.empty:
                            previous_abone_sayisi = previous_hucre[abone_tipi].values[0]
                        else:
                            previous_abone_sayisi = 0
                        
                        # Hücre reel yük hesapla
                        hucre_reel_yuk = abone_basina_reel_yuk * current_abone_sayisi
                        
                        # Hücre vertical yük hesapla
                        hucre_vertical_yuk = abone_basina_vertical_yuk * previous_abone_sayisi
                        
                        # Toplam hücre yükü
                        toplam_hucre_yuku = hucre_reel_yuk + hucre_vertical_yuk
                        
                        # Sonuçları kaydet
                        sonuc_df.loc[hucre_idx, abone_tipi] = toplam_hucre_yuku
            
            # Sonuç dataframe'ini Excel'e yaz
            sonuc_df.to_excel(writer, sheet_name=str(yil), index=False)
            
        except Exception as e:
            print(f"{yil} yılı için hesaplama sırasında hata oluştu: {e}")
    
    # Excel dosyasını kaydet
    writer.close()
    print(f"Hesaplamalar tamamlandı. Sonuçlar {CIKTI_DOSYASI} dosyasına kaydedildi.")

if __name__ == "__main__":
    yuk_hesaplama()

Yük hesaplama başlatılıyor...
2024 yılı için hesaplama yapılıyor...
2025 yılı için hesaplama yapılıyor...
2026 yılı için hesaplama yapılıyor...
2027 yılı için hesaplama yapılıyor...
2028 yılı için hesaplama yapılıyor...
2029 yılı için hesaplama yapılıyor...
2030 yılı için hesaplama yapılıyor...
2031 yılı için hesaplama yapılıyor...
2032 yılı için hesaplama yapılıyor...
2033 yılı için hesaplama yapılıyor...
2034 yılı için hesaplama yapılıyor...
2035 yılı için hesaplama yapılıyor...
Hesaplamalar tamamlandı. Sonuçlar C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\SLF_YUK_DAGITIMI\yukler_sonuc1.xlsx dosyasına kaydedildi.
